# APKGC — Chạy lại thực nghiệm (AAAI 2025)

Seminar GraphML · HP3

**Trước khi chạy:** Runtime → Change runtime type → **T4 GPU**

Notebook tự vá mã nguồn để chọn thiết bị linh hoạt và ghi kết quả vào trong thư mục dự án.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 2. Tải mã nguồn và cài thư viện

In [ ]:
!git clone https://github.com/HubuKG/APKGC.git
%cd APKGC
!pip install -q transformers==4.21.3 easydict unidecode tensorboard

## 3. Áp bản vá tương thích thiết bị

Mã gốc gọi `.cuda()` cứng ở 17 vị trí và ghi kết quả ra `../../data/` — nằm ngoài thư mục dự án. Hai ô dưới sửa cả hai vấn đề.

In [ ]:
%%writefile mmkgc/device.py
import os
import torch

_forced = os.environ.get('APKGC_DEVICE', '').strip().lower()

if _forced:
    DEVICE = torch.device(_forced)
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

def to_device(x):
    return x.to(DEVICE)

def device_name():
    return str(DEVICE)


In [ ]:
import os.path as osp

# --- APKGC.py ---
p = 'mmkgc/module/model/APKGC.py'
s = open(p, encoding='utf-8').read()
if 'from ...device import DEVICE' not in s:
    s = s.replace('from .Model import Model',
                  'from .Model import Model\nfrom ...device import DEVICE')
n = s.count('.cuda()')
s = s.replace('.cuda()', '.to( DEVICE )')
open(p, 'w', encoding='utf-8').write(s)
print('APKGC.py: thay', n, 'cho')

# --- Trainer.py ---
p = 'mmkgc/config/Trainer.py'
s = open(p, encoding='utf-8').read()
if 'from mmkgc.device import DEVICE' not in s:
    s = s.replace('from mmkgc.config import Tester',
                  'from mmkgc.config import Tester\nfrom mmkgc.device import DEVICE')
s = s.replace('self.model.cuda()', 'self.model.to(DEVICE)')
s = s.replace('torch.from_numpy(x).cuda()', 'torch.from_numpy(x).to(DEVICE)')
open(p, 'w', encoding='utf-8').write(s)
print('Trainer.py: xong')

# --- Tester.py ---
p = 'mmkgc/config/Tester.py'
s = open(p, encoding='utf-8').read()
if 'from mmkgc.device import DEVICE' not in s:
    lines = s.split('\n')
    last = max(i for i, l in enumerate(lines) if l.startswith(('import ', 'from ')))
    lines.insert(last + 1, 'from mmkgc.device import DEVICE')
    s = '\n'.join(lines)
s = s.replace('self.model.cuda()', 'self.model.to(DEVICE)')
s = s.replace('torch.from_numpy(x).cuda()', 'torch.from_numpy(x).to(DEVICE)')
open(p, 'w', encoding='utf-8').write(s)
print('Tester.py: xong')

# --- metric.py ---
p = 'torchlight/metric.py'
s = open(p, encoding='utf-8').read()
s = s.replace('dtype=torch.float).cuda()', 'dtype=torch.float, device=predicted.device)')
open(p, 'w', encoding='utf-8').write(s)
print('metric.py: xong')

# --- entrypoints ---
for fn in ['run_KGC_train.py', 'run_KGC_test.py']:
    s = open(fn, encoding='utf-8').read()
    s = s.replace('    torch.cuda.manual_seed_all(args.seed)',
                  '    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(args.seed)')
    s = s.replace("    data_root = osp.abspath(osp.join(this_dir, '..', '..', 'data', ''))",
                  "    data_root = osp.abspath(osp.join(this_dir, 'output'))")
    if 'from mmkgc.device import DEVICE' not in s:
        s = s.replace('from args import get_args',
                      'from args import get_args\nfrom mmkgc.device import DEVICE, device_name')
    s = s.replace('use_gpu=True', "use_gpu=(DEVICE.type != 'cpu')")
    open(fn, 'w', encoding='utf-8').write(s)
    print(fn, ': xong')


In [ ]:
!grep -rn '\.cuda()' --include='*.py' . || echo 'SACH — khong con .cuda() nao'

## 4. Tải bộ nhúng

Nhúng ảnh (BEiT) và văn bản (BERT) đã trích sẵn, không nằm trong repo.

Nguồn: https://drive.google.com/drive/folders/1wGvlzt0i6RYcMniwooFcLKVEs2NBMcpB

In [ ]:
!pip install -q gdown
!gdown --folder 1wGvlzt0i6RYcMniwooFcLKVEs2NBMcpB -O ./embeddings_dl

import shutil, pathlib
pathlib.Path('embeddings').mkdir(exist_ok=True)
moved = 0
for f in pathlib.Path('embeddings_dl').rglob('*'):
    if f.suffix in ('.pth', '.json'):
        dst = pathlib.Path('embeddings') / f.name
        if not dst.exists():
            shutil.move(str(f), str(dst)); moved += 1
print('Da chuyen', moved, 'file vao embeddings/')
!ls -la embeddings/


**Nếu ô trên lỗi** (Drive chặn tải tự động), tải thủ công rồi kéo file vào `APKGC/embeddings/` qua thanh Files bên trái. Cần có:

- `DB15K-visual.pth`, `DB15K-textual.pth`
- `MKG-W-visual.pth`, `MKG-W-textual.pth`
- `missing.json` — bắt buộc cho MKG-W

## 5. Huấn luyện

Chạy thử 300 epoch trước để kiểm tra pipeline thông suốt, rồi mới chạy đầy đủ.

In [ ]:
# CHAY THU NHANH — khoang 10-15 phut
!python run_KGC_train.py \
  -dataset=MKG-W \
  -num_proj=1 -use_intermediate=0 -joint_way=Mformer_hd_mean \
  -batch_size=1024 -margin=12 -dim=128 -neg_num=32 \
  -learning_rate=1e-4 -epoch=300 \
  -noise_update=epoch -noise_ratio=0.2 -mask_ratio=0.7 \
  -use_pool=1 -add_noise=1 \
  -num_hidden_layers=1 -num_attention_heads=2 \
  -save=MKG-W-smoke -exp_id=smoke

In [ ]:
# CHAY DAY DU — cau hinh goc cua tac gia, 8000 epoch, ca hai tap
!bash run.sh

## 6. Chạy nhiều seed — phần bài báo còn thiếu

Bài báo không báo cáo độ lệch chuẩn. Khoảng cách với NativE trên DB15K chỉ 0.002 MRR, nên rất đáng kiểm tra xem con số đó có vượt được dao động giữa các seed hay không.

Đây là đóng góp riêng, nên ghi vào báo cáo.

In [ ]:
import subprocess, re, statistics

SEEDS = [42, 1234, 2025]
EPOCH = 2000   # tang len 8000 neu du thoi gian
results = []

for sd in SEEDS:
    print('\n===== seed', sd, '=====')
    cmd = [
        'python', 'run_KGC_train.py',
        '-dataset=MKG-W', '-num_proj=1', '-use_intermediate=0',
        '-joint_way=Mformer_hd_mean', '-batch_size=1024', '-margin=12',
        '-dim=128', '-neg_num=32', '-learning_rate=1e-4',
        '-epoch=' + str(EPOCH), '-noise_update=epoch', '-noise_ratio=0.2',
        '-mask_ratio=0.7', '-use_pool=1', '-add_noise=1',
        '-num_hidden_layers=1', '-num_attention_heads=2',
        '-seed=' + str(sd), '-save=MKG-W-seed' + str(sd), '-exp_id=seed' + str(sd),
    ]
    out = subprocess.run(cmd, capture_output=True, text=True)
    tail = out.stdout[-3000:]
    print(tail[-800:])
    m = re.findall(r'mrr:([\d.]+)', tail)
    if m:
        results.append((sd, float(m[-1])))

print('\n===== TONG HOP =====')
for sd, v in results:
    print('  seed', sd, ': MRR =', round(v, 4))
if len(results) > 1:
    vals = [v for _, v in results]
    print('\n  trung binh     =', round(statistics.mean(vals), 4))
    print('  do lech chuan  =', round(statistics.stdev(vals), 4))
    print('  bai bao cong bo = 0.3740')


## 7. Lưu kết quả về Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/APKGC_ketqua
!cp -r output/* /content/drive/MyDrive/APKGC_ketqua/ 2>/dev/null
!ls -R /content/drive/MyDrive/APKGC_ketqua | head -40